# SwiGLU Activation
*The gated feedforward activation in LLaMA, PaLM, Gemma, and most modern LLMs.*

**Companion kernels:** `v0_naive.py` → `sm89_v1_vec4_load.py`


## What Is SwiGLU?

**In plain English:** SwiGLU is a "gating" activation function. The feedforward layer produces two parallel outputs: a "gate" vector and an "up" vector. The gate is passed through SiLU (a smooth activation), then element-wise multiplied with the up vector. The gate acts like a learned dimmer switch — it controls how much of each feature passes through.

**Why it matters:** The original transformer used ReLU. SwiGLU (Noam Shazeer, 2020) consistently outperforms both GELU and ReLU at the same parameter count, with no theoretical justification — it just works.


In [ ]:
import math
print('ready')

## Building Blocks: SiLU and GLU

### SiLU (Sigmoid Linear Unit)

$$\text{SiLU}(x) = x \cdot \sigma(x) = \frac{x}{1 + e^{-x}}$$

- When $x \gg 0$: $\sigma(x) \to 1$, so $\text{SiLU}(x) \approx x$ (identity — passes through)
- When $x \approx 0$: $\sigma(0) = 0.5$, so $\text{SiLU}(0) = 0$ (attenuated)
- When $x \ll 0$: $\sigma(x) \to 0$, so $\text{SiLU}(x) \approx 0$ (suppressed)

### GLU (Gated Linear Unit, Dauphin et al. 2017)

$$\text{GLU}(a, b) = a \otimes \sigma(b)$$

SwiGLU replaces $\sigma$ with SiLU:

$$\boxed{\text{SwiGLU}(x, y) = \text{SiLU}(x) \otimes y = \frac{x}{1+e^{-x}} \otimes y}$$


In [ ]:
def silu(x):
    return x / (1.0 + math.exp(-x))

def swiglu(gate, up):
    return silu(gate) * up

# Trace a single element
print("SiLU behavior:")
for x in [-3.0, -1.0, 0.0, 1.0, 3.0]:
    s = silu(x)
    print(f"  SiLU({x:+.1f}) = {s:+.4f}  "
          f"({'≈x' if abs(s-x)<0.1 else '≈0' if abs(s)<0.1 else ''})")

print()
# Element-wise example
gate = [2.0, -1.0, 0.5, 3.0]
up   = [1.5,  2.0, 0.8, 0.3]
print("SwiGLU element-wise:")
print(f"  gate:      {gate}")
print(f"  SiLU(gate): {[round(silu(g),3) for g in gate]}")
out = [swiglu(g, u) for g, u in zip(gate, up)]
print(f"  × up:      {[round(v,3) for v in out]}")


## The Formula

$$\boxed{\text{SwiGLU}(x, y) = \text{SiLU}(x) \cdot y = \frac{x}{1 + e^{-x}} \cdot y}$$

| Symbol | Meaning |
|--------|---------|
| $x$ | Gate vector (from one linear projection of the input) |
| $y$ | Up-projection vector (from another linear projection) |
| $\sigma(x) = \frac{1}{1+e^{-x}}$ | Sigmoid — squashes $x$ to $(0,1)$ |
| $\text{SiLU}(x) = x \cdot \sigma(x)$ | Smooth, differentiable, no "dead neuron" problem |
| $\otimes$ | Element-wise multiplication |

### 🗣️ Say It Out Loud
> *"SwiGLU of x and y equals x times sigma of x, times y — where sigma is the sigmoid function, one over one plus e to the minus x."*

**Tutor's gloss:** "Think of $x$ as the gate controlling a valve and $y$ as the flow through it. When the gate $x$ is large positive, $\sigma(x) \approx 1$, so the valve is fully open: $y$ passes through unchanged. When $x$ is near zero, the valve is half-open. When $x$ is large negative, it's nearly closed: $y$ is suppressed. The model learns to set the gate values to route information."


## SwiGLU in Context: The LLaMA FFN

A standard transformer FFN layer:
$$\text{FFN}(x) = W_\text{down} \cdot \text{ReLU}(W_\text{up} \cdot x)$$

LLaMA's SwiGLU FFN:
$$\text{FFN}_\text{SwiGLU}(x) = W_\text{down} \cdot \bigl(\text{SiLU}(W_\text{gate} \cdot x) \otimes W_\text{up} \cdot x\bigr)$$

Three weight matrices instead of two. To keep the same parameter count, LLaMA uses:

$$d_\text{hidden} = \left\lfloor \frac{8}{3} d_\text{model} \right\rfloor \approx 2.67 \times d_\text{model}$$

instead of the usual $4 \times d_\text{model}$.

| Variant | Activation | Weight matrices | Typical hidden dim |
|---------|-----------|-----------------|-------------------|
| Original Transformer | ReLU | 2 | $4d$ |
| GPT-2/3 | GELU | 2 | $4d$ |
| LLaMA / PaLM | SwiGLU | 3 | $8d/3 \approx 2.67d$ |

**Total parameters equivalent** (at $d=4096$): $2 \times 4096 \times 16384 = 2 \times 4096 \times 10923$ — same count.


## Optimization Ladder

| Version | Strategy | Key idea |
|---------|----------|----------|
| `v0_naive` | One thread per element, scalar loads | Establishes correctness baseline |
| `sm89_v1_vec4_load` | Vectorized `float4` loads | 4 BF16 values per load instruction — hits bandwidth ceiling |

**Why SwiGLU barely needs optimizing:**
The activation itself (SiLU × up) does ~3 FLOPs per element at 4 bytes in/out — barely above RMSNorm's ratio.
But it doesn't matter: the upstream GEMMs (W_gate·x and W_up·x) are 100–1000× more expensive.
The activation is always the cheapest part of the FFN forward pass.

## CuTeDSL: Fused Gate × Up Kernel

```python
@cute.kernel
def swiglu_kernel(mGate, mUp, mOut):
    # One thread per output element — no reduction needed
    i = blockIdx.x * blockDim.x + threadIdx.x

    gate = mGate[i]   # result of W_gate · x (one projection)
    up   = mUp[i]     # result of W_up · x (another projection)

    # SiLU(gate) = gate * sigmoid(gate) = gate / (1 + exp(-gate))
    # SM89 tip: cute.exp uses __expf, a fast 4-ulp approximation (~4× faster than full exp)
    silu_val = gate / (1.0 + cute.exp(-gate))

    mOut[i] = silu_val * up   # the "gate": controls how much of `up` passes through
```

**The v1 upgrade — loading 4 values at once:**
```python
# v0: one element per load
gate = mGate[i]    # 1 BF16 = 2 bytes per instruction

# v1: four elements per load  
gate_vec = mGate.load_v4(i)   # 4 BF16 = 8 bytes per instruction, same latency
# Then apply SiLU to each component of gate_vec
```
This directly increases the fraction of peak memory bandwidth actually achieved.

## RTX 4080: The Real Cost is in the Weight GEMMs, Not the Activation

The SwiGLU activation itself: ~3 FLOPs/element, 4 bytes/element → 0.75 FLOP/byte. Memory-bound.
But the activation takes microseconds. The weight GEMMs take milliseconds.

**Qwen3-8B MLP weight sizes:**
- W_gate: [12288, 4096] in BF16 = 100 MB
- W_up:   [12288, 4096] in BF16 = 100 MB
- W_down: [4096, 12288] in BF16 = 100 MB
- Total: 300 MB of weights per MLP layer, times 36 layers = 10.8 GB just for MLP weights

**At decode (M=1, generating one token):**
Reading W_gate + W_up + W_down each forward pass = 300 MB moved across HBM.
At 380 GB/s: 300 MB / 380 GB/s = **0.79 ms per decode step, just in weight reads**.
Tensor cores contribute almost nothing — the GPU waits on memory.

**FP8 quantization changes everything:**
- W_gate + W_up in FP8: 50 MB each (half the size)
- Reads drop from 300 MB → 150 MB per forward pass
- Time drops from 0.79 ms → ~0.40 ms
- SM89 native FP8 tensor cores give another ~2× on the compute side
- Total MLP speedup: roughly **2–3× faster decode** vs BF16

This is the biggest single optimization available on the RTX 4080 for inference: quantize the MLP weights to FP8.

## Suggestion: Fuse SwiGLU into the Down-Projection GEMM

The MLP forward pass currently launches three separate kernels:
```
1. gate_out = x @ W_gate.T   (GEMM)     shape: [M, 12288]  → HBM write: M×12288×2 bytes
2. up_out   = x @ W_up.T     (GEMM)     shape: [M, 12288]  → HBM write: M×12288×2 bytes
3. hidden   = SwiGLU(gate_out, up_out)  shape: [M, 12288]  → HBM read: 2 × M×12288×2
             (reads both gate_out and up_out)                 HBM write: M×12288×2 bytes
4. out      = hidden @ W_down.T (GEMM)  shape: [M, 4096]   → HBM read: M×12288×2 bytes
```

At **prefill (M=2048)**, the intermediate tensors ([2048, 12288] = 48 MB each) move through HBM four times: write gate_out, write up_out, read gate_out+up_out for SwiGLU, read hidden for down_proj.

**Fusion opportunity 1: fuse SwiGLU into gates kernel epilogue**
Instead of writing gate_out and up_out to HBM, compute them both in registers and immediately apply SwiGLU, writing only `hidden` to HBM. This requires running W_gate and W_up GEMMs with a shared thread block that accumulates both outputs simultaneously:

```python
# "Dual GEMM + fused activation" epilogue:
# Thread computes both gate_val and up_val for the same output position
rC_gate  = cute.partition_fragment_C(tiled_mma, ...)   # accumulator for gate
rC_up    = cute.partition_fragment_C(tiled_mma, ...)   # accumulator for up

for k_tile in range(K // Bk):
    # Load input tile (shared between both GEMMs — load once, use twice)
    cute.copy(mX_tile, sX)
    cute.sync_threads()

    cute.copy(mWgate_tile, sWgate)
    cute.gemm(tiled_mma, sX, sWgate, rC_gate)   # gate accumulation

    cute.copy(mWup_tile, sWup)
    cute.gemm(tiled_mma, sX, sWup,   rC_up)     # up accumulation

# Fused epilogue: SwiGLU before writing to HBM
for elem in range(elems_per_thread):
    gate = rC_gate[elem]
    up   = rC_up[elem]
    hidden = gate / (1.0 + cute.exp(-gate)) * up   # SwiGLU in registers
    mHidden[...] = hidden   # write to HBM only once
```

**Bytes saved at prefill M=2048:**
```
Without fusion: write gate (48 MB) + write up (48 MB) + read gate+up (96 MB) = 192 MB extra
With fusion:    write hidden only (48 MB)
Saved: 144 MB × 380 GB/s ≈ 0.38 ms per MLP layer per call
```

**Fusion opportunity 2: fuse hidden directly into W_down GEMM input**
Take it further: don't write `hidden` to HBM at all. Instead, stream SwiGLU output directly into the W_down GEMM as the A matrix. This is the "persistent kernel" approach used by FasterTransformer and TensorRT-LLM.

At prefill this is complex to implement. At decode (M=1), hidden is only 12,288 values (24 KB) — easily cached in L1, so the separate SwiGLU kernel barely matters.

## Decode Profiling Breakdown for One MLP Layer (Qwen3-8B)

At decode (one output token), here is where the time goes.
Estimates use 380 GB/s bandwidth and 57.5 TFLOPS BF16.

```
Operation           | Data read  | FLOPs      | BW-bound time | Compute-bound time
────────────────────────────────────────────────────────────────────────────────────
gate_proj (GEMM)    | 100 MB     | 33 MFLOPs  |   263 µs      |  0.6 µs
up_proj   (GEMM)    | 100 MB     | 33 MFLOPs  |   263 µs      |  0.6 µs
SwiGLU activation   |  72 MB     | 36 MFLOPs  |   190 µs      |  trivial
down_proj (GEMM)    | 100 MB     | 100 MFLOPs |   263 µs      |  1.7 µs
────────────────────────────────────────────────────────────────────────────────────
MLP total           | 372 MB     | 202 MFLOPs |  ~979 µs      |  ~3.5 µs
```

At decode: **memory-bound by 280×**. The compute time is a rounding error.

**The three levers to go faster:**

**1. Weight quantization (FP8):**
Cuts 300 MB of weights to 150 MB → ~490 µs per MLP layer (~2× faster).

**2. Larger batch size:**
At batch_size=8, all 8 tokens share the same weight reads.
The 300 MB is still read once, but now produces 8 outputs instead of 1.
Effective bandwidth per token: 300 MB ÷ 8 = 37.5 MB → ~99 µs per token.
Still memory-bound, but 8× more efficient.

**3. FP8 + batching together:**
150 MB weights ÷ 8 tokens → ~18 MB per token → approaching compute-bound territory.

**When does MLP become compute-bound?**
For down_proj at batch size M:
- FLOPs = 2 × M × 4096 × 12288
- Bytes ≈ M × 12288 × 2 (input) + 4096 × 12288 × 2 (weights) + M × 4096 × 2 (output)
- Crossover (intensity > 151 F/B) happens around **M ≈ 5**

Even at batch_size=5, the MLP is compute-bound. Weight quantization and batch size
are the two knobs worth reaching for on the RTX 4080.